# Demo histórica del NPP-LSTM

## Objetivo

Permitir una evaluación interactiva del modelo NPP
Softplus W20 frente a Poisson.

El usuario podrá:
- Seleccionar una referencia histórica.
- Elegir un horizonte de 1, 7 o 30 días.
- Comparar probabilidades estimadas.
- Consultar si realmente ocurrió un evento Mw >= 4.5.

La aplicación reutiliza los pesos entrenados y
el catálogo IGP.

**Advertencia:** Es un prototipo académico para
evaluación retrospectiva, no un sistema de alerta sísmica.

In [4]:

from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

import sys
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import gradio as gr

raiz = Path.cwd()
if not (raiz / "data").exists():
    raiz = raiz.parent

sys.path.insert(0, str(raiz / "src"))

from utils import (
    datetime_to_hours,
    truncate_catalog_by_threshold,
    find_Poisson_MLE
)
from npp_model import construir_npp

VENTANA = 20
MCUT = 3.5
M0PRED = 4.5

# 1. Leer el catálogo sísmico
catalogo = pd.read_csv(
    raiz / "data/raw/SISMOS_LIMA_FORMATO_MODELO_cat.csv"
)

catalogo = datetime_to_hours(catalogo).dropna()
catalogo = truncate_catalog_by_threshold(catalogo, MCUT)
catalogo = catalogo.reset_index(drop=True)

fechas = pd.to_datetime(
    catalogo[[
        "year", "month", "day",
        "hour", "minute", "second"
    ]]
)

tiempos = catalogo["time"].to_numpy()
magnitudes = catalogo["mw"].to_numpy()

assert fechas.is_monotonic_increasing
assert np.all(np.diff(tiempos) > 0)

# 2. Mantener la separación original
T_train = tiempos[tiempos < 450000]
M_train = magnitudes[tiempos < 450000]

indice_inicio_test = np.searchsorted(
    tiempos, 450000
)

inicio_test = fechas.iloc[indice_inicio_test]
ultimo_registro = fechas.iloc[-1]

# 3. Estadísticas del entrenamiento W20
mu_t = np.log(np.diff(T_train)).mean()
sigma_t = np.log(np.diff(T_train)).std()

mu_m = np.log(M_train).mean()
sigma_m = np.log(M_train).std()

# 4. Construir y recuperar la red entrenada
modelo = construir_npp(
    ventana=VENTANA,
    mu_t=mu_t,
    sigma_t=sigma_t,
    mu_m=mu_m,
    sigma_m=sigma_m,
    m0pred=M0PRED,
    activacion_magnitud="softplus"
)

modelo.load_weights(
    str(raiz / "models/NPP_softplus_W20.weights.h5")
)

# 5. Recuperar el modelo Poisson original
tasa_poisson = float(
    find_Poisson_MLE(
        T_train, M_train, M0PRED
    )
)

# Eventos para comprobar resultados históricos
fechas_objetivo = fechas[
    magnitudes >= M0PRED
].to_numpy()

print("NPP Softplus W20 cargado correctamente.")
print("Tasa Poisson por hora:", tasa_poisson)
print("Inicio del test:", inicio_test)
print("Último registro del catálogo:", ultimo_registro)
print("Gradio:", gr.__version__)


NPP Softplus W20 cargado correctamente.
Tasa Poisson por hora: 0.003581180843539231
Inicio del test: 2012-05-31 06:35:55
Último registro del catálogo: 2026-11-02 15:28:08
Gradio: 6.31.0


In [5]:

def construir_tarjeta(nombre, probabilidad, descripcion, color):
    porcentaje = f"{probabilidad * 100:.2f}%"

    return f"""
    <div style="
        background:{color};
        padding:24px;
        border-radius:16px;
        color:white;
        min-width:210px;
        flex:1;
        text-align:center;
    ">
        <div style="font-size:16px;font-weight:600;">
            {nombre}
        </div>
        <div style="
            font-size:clamp(40px,6vw,66px);
            font-weight:800;
            margin:10px 0;
        ">
            {porcentaje}
        </div>
        <div style="font-size:13px;opacity:0.9;">
            {descripcion}
        </div>
    </div>
    """


def evaluar_fecha(modo, fecha_elegida, dias):

    # A. Determinar la fecha de consulta
    if modo == "Desde hoy":
        referencia = pd.Timestamp(
            datetime.now(ZoneInfo("America/Lima")).replace(
                tzinfo=None
            )
        )
    else:
        if not fecha_elegida:
            raise gr.Error("Selecciona una fecha histórica.")

        referencia = pd.Timestamp(
            pd.to_datetime(fecha_elegida).date()
        )

        if referencia < inicio_test:
            raise gr.Error(
                "Selecciona una fecha posterior al inicio "
                "del conjunto de prueba: "
                f"{inicio_test.date()}."
            )

        hoy = pd.Timestamp(
            datetime.now(ZoneInfo("America/Lima")).date()
        )

        if referencia > hoy:
            raise gr.Error(
                "La fecha histórica no puede ser futura."
            )

    dias = int(dias)
    horas = dias * 24

    # B. Localizar el último evento conocido
    indice = np.searchsorted(
        fechas.to_numpy(),
        referencia.to_datetime64(),
        side="right"
    ) - 1

    if indice < VENTANA:
        raise gr.Error(
            "No existen suficientes eventos anteriores."
        )

    # C. Recuperar exactamente 20 intervalos
    hist_t = np.diff(
        tiempos[indice - VENTANA:indice + 1]
    )

    hist_m = magnitudes[
        indice - VENTANA + 1:indice + 1
    ]

    hist_t = tf.convert_to_tensor(
        hist_t.reshape(1, VENTANA, 1),
        dtype=tf.float32
    )

    hist_m = tf.convert_to_tensor(
        hist_m.reshape(1, VENTANA, 1),
        dtype=tf.float32
    )

    # D. Horas desde el último evento hasta la consulta
    ultimo_evento = fechas.iloc[indice]

    transcurridas = (
        referencia - ultimo_evento
    ).total_seconds() / 3600.0

    # E. Riesgo acumulado condicional aproximado
    t_inicial = max(transcurridas, 1e-3)
    t_final = transcurridas + horas

    mag_aux = tf.constant([[4.5]], dtype=tf.float32)

    H_inicio, _ = modelo(
        [
            hist_t, hist_m,
            tf.constant([[t_inicial]], dtype=tf.float32),
            mag_aux
        ],
        training=False
    )

    H_final, _ = modelo(
        [
            hist_t, hist_m,
            tf.constant([[t_final]], dtype=tf.float32),
            mag_aux
        ],
        training=False
    )

    diferencia = float(
        H_final.numpy()[0, 0] -
        H_inicio.numpy()[0, 0]
    )

    if not np.isfinite(diferencia) or diferencia < -1e-6:
        raise gr.Error(
            "La función temporal produjo un riesgo "
            "acumulado inválido para esta fecha."
        )

    delta_H = max(diferencia, 0.0)

    p_npp = float(-np.expm1(-delta_H))
    p_poisson = float(
        -np.expm1(-tasa_poisson * horas)
    )

    # F. Tarjetas de porcentajes grandes
    tarjetas = f"""
    <div style="
        display:flex;
        flex-wrap:wrap;
        gap:16px;
        margin:12px 0;
    ">
        {construir_tarjeta(
            "NPP-LSTM Softplus W20",
            p_npp,
            "Estimación neuronal experimental",
            "#153653"
        )}
        {construir_tarjeta(
            "Poisson",
            p_poisson,
            "Referencia estadística",
            "#116B5B"
        )}
    </div>
    """

    # G. Verificar los acontecimientos históricos
    fin_periodo = referencia + pd.Timedelta(days=dias)

    if (
        modo == "Histórico"
        and fin_periodo <= ultimo_registro
    ):
        inicio = np.searchsorted(
            fechas_objetivo,
            referencia.to_datetime64(),
            side="right"
        )

        fin = np.searchsorted(
            fechas_objetivo,
            fin_periodo.to_datetime64(),
            side="right"
        )

        cantidad = int(fin - inicio)

        if cantidad > 0:
            observacion = (
                f"✅ Según el catálogo, ocurrieron "
                f"**{cantidad} eventos Mw ≥ 4.5** "
                "en ese periodo."
            )
        else:
            observacion = (
                "❌ Según el catálogo, no se registraron "
                "eventos Mw ≥ 4.5 en ese periodo."
            )
    else:
        observacion = (
            "⏳ **Resultado no verificable todavía.** "
            "El catálogo disponible no cubre "
            "completamente el periodo consultado."
        )

    advertencia = ""
    if referencia > ultimo_registro:
        advertencia = (
            "\n\n⚠️ **Datos potencialmente desactualizados:** "
            f"el último sismo del catálogo es del "
            f"{ultimo_registro:%d/%m/%Y %H:%M}. "
            "La estimación supone que no faltan eventos "
            "posteriores que alteren el historial del modelo."
        )

    descripcion = f"""
### Resultado de la consulta

**Modo:** {modo}

**Fecha de referencia:** {referencia:%d/%m/%Y %H:%M}

**Último sismo utilizado:** {ultimo_evento:%d/%m/%Y %H:%M}

**Tiempo transcurrido desde ese sismo:** {transcurridas:.1f} horas

**Horizonte:** {dias} días

**Magnitud objetivo:** Mw ≥ 4.5

### ¿Qué ocurrió realmente?

{observacion}

{advertencia}

**Importante:** Los porcentajes son estimaciones
experimentales no calibradas. No constituyen
pronósticos operativos ni alertas sísmicas.
"""

    # H. Gráfico comparativo
    fig, ax = plt.subplots(figsize=(8, 4))

    nombres = ["NPP-LSTM", "Poisson"]
    valores = [p_npp * 100, p_poisson * 100]

    barras = ax.bar(
        nombres,
        valores,
        color=["#2467AB", "#15977C"]
    )

    ax.set_ylim(0, 110)
    ax.set_ylabel("Probabilidad estimada (%)")
    ax.set_title(f"Horizonte de {dias} días")

    for barra, valor in zip(barras, valores):
        ax.text(
            barra.get_x() + barra.get_width() / 2,
            valor + 2,
            f"{valor:.2f}%",
            ha="center",
            fontweight="bold"
        )

    plt.tight_layout()
    plt.close(fig)

    return tarjetas, descripcion, fig


In [6]:

with gr.Blocks(
    title="NPP-Lima | Demo sísmica"
) as demo:

    gr.Markdown("""
    # NPP-Lima 🧠
    ## Estimación experimental de ocurrencia sísmica

    Comparación entre **NPP-LSTM Softplus W20**
    y el modelo **Poisson**.

    Ambos estiman la ocurrencia de al menos un
    evento de magnitud **Mw ≥ 4.5**.

    **Prototipo académico. No es un sistema
    de alerta sísmica.**
    """)

    with gr.Row():
        modo = gr.Radio(
            choices=["Histórico", "Desde hoy"],
            value="Histórico",
            label="Tipo de consulta"
        )

        dias = gr.Radio(
            choices=[1, 7, 30],
            value=7,
            label="Horizonte en días"
        )

    fecha = gr.DateTime(
        value="2026-10-01",
        include_time=False,
        type="string",
        label="Selecciona una fecha histórica"
    )

    modo.change(
        fn=lambda m: gr.update(
            visible=(m == "Histórico")
        ),
        inputs=modo,
        outputs=fecha
    )

    boton = gr.Button(
        "Calcular estimación",
        variant="primary"
    )

    gr.Markdown("## Probabilidades estimadas")

    tarjetas = gr.HTML()
    descripcion = gr.Markdown()
    grafico = gr.Plot()

    boton.click(
        fn=evaluar_fecha,
        inputs=[modo, fecha, dias],
        outputs=[tarjetas, descripcion, grafico]
    )

    demo.load(
        fn=evaluar_fecha,
        inputs=[modo, fecha, dias],
        outputs=[tarjetas, descripcion, grafico]
    )

demo.launch(inbrowser=True, share=False)


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
